# Exploring Radio Signals and Classifying Fashion Images
**DASC 3230 · Individual project 3 · Weeks 7–9**

Explore structure in radio-signal candidates, screen candidates against a
background reference, and build a small image classifier.

Complete **12 TODOs**, **three code-reading responses (S1–S3)**, and
**three result responses (R1–R3)**. Keep written answers brief and refer to
your code or displayed results.

Use NumPy, Matplotlib, PyTorch, and only PCA and KernelDensity from scikit-learn.
Implement the K-means assignment and centre update with NumPy.
Keep the supplied data, seeds, settings, and checks. Replace each
NotImplementedError with your code.

| Part | Topic | Marks |
|---|---|---:|
| A | PCA and K-means | 35 |
| B | Density estimation and candidate screening | 30 |
| C | Image classification with a CNN | 35 |
| **Total** | | **100** |

The supplied helpers handle data splitting, cross-validation, training,
and most plotting. Marks are based on your implementation and explanations;
there is no minimum accuracy requirement.

See [data/README.pdf](data/README.pdf) for data sources and feature descriptions.
Submit one A4 PDF containing the completed notebook, code, outputs, and answers.


In [ ]:
from pathlib import Path
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.neighbors import KernelDensity

SEED = 3230
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
plt.rcParams.update({"font.size": 10, "figure.dpi": 100})

PROJECT_DIR = Path.cwd()
print(PROJECT_DIR)
if not (PROJECT_DIR / "data" / "htru2_small.npz").exists():
    PROJECT_DIR = PROJECT_DIR.parent / "individual_project_3"
DATA_DIR = PROJECT_DIR / "data"
assert DATA_DIR.is_dir(), "Open the notebook from its project folder."

def fit_standardizer(X):
    mean = X.mean(axis=0)
    scale = X.std(axis=0, ddof=0)
    scale = np.where(scale == 0, 1., scale)
    return mean, scale

def apply_standardizer(X, mean, scale):
    return (X - mean) / scale


## A. HTRU2: structure in radio candidates — 35 marks

Each row describes one radio-signal candidate. We use four features:
the mean and standard deviation of its pulse profile, and the mean and
standard deviation of its DM–SNR curve. These summaries are already computed.

For PCA and clustering, use the mixed training sample. The class labels
are shown later to help interpret the groups: 0 is a background candidate
(non-pulsar), and 1 is a pulsar candidate.


In [ ]:
with np.load(DATA_DIR / "htru2_small.npz", allow_pickle=False) as data:
    radio_X_train = data["X_train"]
    radio_y_train = data["y_train"]
assert radio_X_train.shape == (1650, 4)
assert radio_y_train.shape == (1650,)
explore_mean, explore_scale = fit_standardizer(radio_X_train)
radio_U_train = apply_standardizer(radio_X_train, explore_mean, explore_scale)
print("Training observations and features:", radio_U_train.shape)


### Two PCA coordinates
**TODO 1 (6 marks):** Fit a two-component PCA to U, which has already been standardized by the supplied code.
U has shape (n, p).
**Return (pca, Z, retained), in this order:**
the fitted PCA object, a NumPy score array of shape (n, 2), and the scalar
sum of the two explained variance ratios.


In [ ]:
def fit_pca(U):
    # TODO 1
    raise NotImplementedError("TODO 1")

toy_U = np.array([[-2., -2.], [-1., -1.], [1., 1.], [2., 2.]])
toy_pca, toy_Z, toy_retained = fit_pca(toy_U)
assert toy_Z.shape == (4, 2)
assert toy_pca.components_.shape == (2, 2)
assert np.isclose(toy_retained, 1.0)

radio_pca, radio_Z_train, radio_retained = fit_pca(radio_U_train)
assert radio_Z_train.shape == (1650, 2)
print("Variance ratios:", np.round(radio_pca.explained_variance_ratio_, 4))
print(f"Retained variance: {radio_retained:.2%}")


**TODO 2 (5 marks):** reconstruct the standardized features from their PCA
scores and calculate the reconstruction MSE.
Given U of shape (n, p), scores Z of shape (n, 2), and the fitted pca,
**return (rebuilt, error)**.
rebuilt is the NumPy reconstruction of shape (n, p); error is its scalar
mean squared error over all entries of U. Use standardized feature units.


In [ ]:
def reconstruct_pca(U, Z, pca):
    # TODO 2
    raise NotImplementedError("TODO 2")

toy_rebuilt, toy_error = reconstruct_pca(toy_U, toy_Z, toy_pca)
assert np.allclose(toy_rebuilt, toy_U) and np.isclose(toy_error, 0)
partial_pca = PCA(n_components=2).fit(np.vstack([np.eye(3), -np.eye(3)]))
check_U = np.vstack([np.eye(3), -np.eye(3)])
_, partial_error = reconstruct_pca(check_U, partial_pca.transform(check_U), partial_pca)
assert np.isclose(partial_error, 1 / 9)

radio_U_rebuilt, radio_reconstruction_mse = reconstruct_pca(
    radio_U_train, radio_Z_train, radio_pca)
assert radio_U_rebuilt.shape == radio_U_train.shape
print(f"Standardized reconstruction MSE: {radio_reconstruction_mse:.4f}")


### K-means on four standardized features
**TODO 3 (6 marks):** assign each observation to its nearest centroid using
squared Euclidean distance.
Given U of shape (n, p) and centres of shape (K, p), **return labels**,
an integer NumPy array of shape (n,) containing centroid indices from 0 to K - 1.
In a tie, choose the first centroid.

**TODO 4 (6 marks):** update each cluster centre using the mean of its assigned
observations.
Given U of shape (n, p), labels of shape (n,), and old_centres of shape (K, p),
**return a new NumPy array of shape (K, p)** containing the updated centres.
Keep the previous centre for an empty cluster, and leave old_centres unchanged.


In [ ]:
def assign_clusters(U, centres):
    # TODO 3
    raise NotImplementedError("TODO 3")

check_U = np.array([[0., 0.], [2., 0.], [4., 2.]])
check_centres = np.array([[0., 0.], [4., 0.]])
assert np.array_equal(assign_clusters(check_U, check_centres), [0, 0, 1])
assert assign_clusters(check_U, check_centres).shape == (3,)
print("Assignment check passed.")


In [ ]:
def update_centres(U, labels, old_centres):
    # TODO 4
    raise NotImplementedError("TODO 4")

check_old = np.array([[9., 9.], [4., 0.], [-3., 5.]])
check_new = update_centres(check_U, np.array([0, 0, 1]), check_old)
assert np.allclose(check_new, [[1., 0.], [4., 2.], [-3., 5.]])
assert np.allclose(check_old, [[9., 9.], [4., 0.], [-3., 5.]])
print("Centre update check passed.")


The supplied driver runs your two steps from ten fixed initializations and
keeps the result with the smallest inertia. K is fixed at 2.
PCA supplies the coordinates for the display; clustering uses all four features.


In [ ]:
def run_kmeans(U, K=2, seed=SEED):
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(10):
        centres = U[rng.choice(len(U), K, replace=False)].copy()
        for _ in range(100):
            labels = assign_clusters(U, centres)
            updated = update_centres(U, labels, centres)
            converged = np.allclose(updated, centres, rtol=0, atol=1e-8)
            centres = updated
            if converged:
                break
        labels = assign_clusters(U, centres)
        inertia = float(np.sum((U - centres[labels]) ** 2))
        if best is None or inertia < best["inertia"]:
            best = {"centres": centres.copy(), "labels": labels.copy(),
                    "inertia": inertia}
    return best

radio_clusters = run_kmeans(radio_U_train)
cluster_table = []
print("Cluster  Background  Pulsar  Total")
for k in range(2):
    mask = radio_clusters["labels"] == k
    row = {"cluster": k, "background": int(np.sum(radio_y_train[mask] == 0)),
           "pulsar": int(np.sum(radio_y_train[mask] == 1)), "total": int(mask.sum())}
    cluster_table.append(row)
    print(f'{k:7d} {row["background"]:11d} {row["pulsar"]:7d} {row["total"]:6d}')

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for ax, colour, title in zip(
        axes, [radio_clusters["labels"], radio_y_train],
        ["K-means clusters", "Observed labels: 0 background, 1 pulsar"]):
    ax.scatter(radio_Z_train[:, 0], radio_Z_train[:, 1], c=colour,
               cmap="coolwarm", s=9, alpha=.6)
    ax.set(xlabel="PC1 score", ylabel="PC2 score", title=title)
fig.tight_layout()
plt.show()


**S1 (5 marks).**

(a) What do the rows and two columns of radio_Z_train represent?
How do these scores differ from the K-means labels?

(b) Why standardize the features before distance-based clustering?

**Answer:**


**R1 (7 marks).**

(a) Report the retained variance and reconstruction MSE.
What do they tell you about using two coordinates instead of four?

(b) Does one cluster contain only background candidates and the other only
pulsar candidates? Use the counts in the table to support your answer.

**Answer:**


## B. HTRU2: screening against a background reference — 30 marks

We now fit a density using only background training candidates.
The two features are pulse-profile mean and standard deviation.
The standardization mean and scale are estimated from this reference sample.

Training folds select the bandwidth. The separate background calibration
sample sets the threshold. Test labels are used for the final evaluation.


In [ ]:
background_X = radio_X_train[radio_y_train == 0, :2]
density_mean, density_scale = fit_standardizer(background_X)
background_U = apply_standardizer(background_X, density_mean, density_scale)
with np.load(DATA_DIR / "htru2_small.npz", allow_pickle=False) as data:
    calibration_U = apply_standardizer(
        data["X_calibration"][:, :2], density_mean, density_scale)
assert background_U.shape == (1500, 2) and calibration_U.shape == (400, 2)

BANDWIDTHS = np.array([0.1, 0.3, 1.0])
indices = np.random.default_rng(SEED).permutation(len(background_X))
validation_folds = np.array_split(indices, 3)
folds = []
for k in range(3):
    val_idx = validation_folds[k]
    fit_idx = np.concatenate(validation_folds[:k] + validation_folds[k + 1:])
    folds.append((fit_idx, val_idx))
cv_log_density = np.empty((len(BANDWIDTHS), len(folds)))
for j, h in enumerate(BANDWIDTHS):
    for k, (fit_idx, val_idx) in enumerate(folds):
        fold_mean, fold_scale = fit_standardizer(background_X[fit_idx])
        fit_U = apply_standardizer(background_X[fit_idx], fold_mean, fold_scale)
        val_U = apply_standardizer(background_X[val_idx], fold_mean, fold_scale)
        fold_kde = KernelDensity(kernel="gaussian", bandwidth=float(h)).fit(fit_U)
        cv_log_density[j, k] = fold_kde.score_samples(val_U).mean()
print("Rows: bandwidths; columns: held-out training folds")
print(np.round(cv_log_density, 4))


**TODO 5 (5 marks):** implement select_bandwidth to choose a KDE bandwidth
using the supplied cross-validation scores. For each bandwidth, average
its held-out log-density scores across folds. Select the bandwidth with
the highest average, choosing the first candidate in an exact tie.

scores has one row per bandwidth and one column per fold.
**Return (means, best_index):** a one-dimensional NumPy array of the mean
scores, followed by the zero-based index of the selected bandwidth.


In [ ]:
def select_bandwidth(scores):
    # TODO 5
    raise NotImplementedError("TODO 5")

check_means, check_best = select_bandwidth(np.array([[-4., -2.], [-2., -1.]]))
assert np.allclose(check_means, [-3., -1.5]) and check_best == 1
assert select_bandwidth(np.ones((3, 2)))[1] == 0
mean_cv_log_density, bandwidth_index = select_bandwidth(cv_log_density)
selected_bandwidth = float(BANDWIDTHS[bandwidth_index])
print("Mean log densities:", np.round(mean_cv_log_density, 4))
print("Selected bandwidth:", selected_bandwidth)


**TODO 6 (6 marks):** fit a Gaussian KernelDensity model on U_reference,
shape (n, p), using the supplied positive bandwidth.
For U_query of shape (m, p), **return (model, scores)**:
the fitted model and a NumPy array of shape (m,) containing negative
log-density scores.


In [ ]:
def fit_and_score_kde(U_reference, U_query, bandwidth):
    # TODO 6
    raise NotImplementedError("TODO 6")

toy_kde, toy_scores = fit_and_score_kde(np.array([[0.]]), np.array([[0.], [1.]]), 1.)
assert np.allclose(toy_scores, [0.5 * np.log(2 * np.pi),
                               0.5 * np.log(2 * np.pi) + 0.5])
assert toy_scores.shape == (2,)
density_model, calibration_scores = fit_and_score_kde(
    background_U, calibration_U, selected_bandwidth)
assert calibration_scores.shape == (400,) and np.isfinite(calibration_scores).all()


**TODO 7 (4 marks):** set the threshold at the 95th percentile of the calibration
scores and flag scores above it.
For calibration scores of shape (n,),
**return (tau, flags)**: the scalar 95th percentile and a Boolean NumPy
array of shape (n,) equal to scores > tau.


In [ ]:
def calibrate_threshold(scores):
    # TODO 7
    raise NotImplementedError("TODO 7")

check_tau, check_flags = calibrate_threshold(np.arange(21, dtype=float))
assert np.isclose(check_tau, 19.) and check_flags.sum() == 1
assert check_flags.dtype == np.bool_
tau, calibration_flags = calibrate_threshold(calibration_scores)
density_selection = {"bandwidth": selected_bandwidth, "threshold": float(tau)}
print("Fixed density choices:", density_selection)
print(f"Background calibration flags: {calibration_flags.sum()}/{len(calibration_flags)}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(BANDWIDTHS, mean_cv_log_density, "o-")
axes[0].set(xlabel="Bandwidth", ylabel="Mean held-out log density",
            title="Bandwidth selection within training data")
axes[1].hist(calibration_scores, bins=22, color="#4978a5", edgecolor="white")
axes[1].axvline(tau, color="#be593d", label="95th-percentile threshold")
axes[1].set(xlabel="Negative log density", ylabel="Calibration candidates")
axes[1].legend()
fig.tight_layout()
plt.show()


**S2 (5 marks).**

(a) Why is a candidate's negative log density not its probability of being a pulsar?

(b) What does the 95th-percentile threshold mean for the background calibration
sample? Does it guarantee a 5% background false-alarm rate on test data?

**Answer:**


### Fixed test evaluation
**TODO 8 (5 marks):** standardize the two test features using density_mean
and density_scale, score them with density_model, and flag scores strictly greater than tau.
Use the supplied detection_summary for the metrics. Keep the selected model
and threshold fixed.

**Define:** radio_test_scores, a NumPy array of shape (900,);
radio_test_flags, a Boolean NumPy array of shape (900,); and
radio_test_metrics, the dictionary returned by detection_summary.


In [ ]:
def detection_summary(actual, flagged):
    actual = np.asarray(actual, dtype=int)
    flagged = np.asarray(flagged, dtype=bool)
    tn = int(np.sum((actual == 0) & ~flagged))
    fp = int(np.sum((actual == 0) & flagged))
    fn = int(np.sum((actual == 1) & ~flagged))
    tp = int(np.sum((actual == 1) & flagged))
    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp,
            "precision": tp / (tp + fp) if tp + fp else 0.,
            "recall": tp / (tp + fn) if tp + fn else 0.,
            "background_false_alarm_rate": fp / (tn + fp) if tn + fp else 0.}

with np.load(DATA_DIR / "htru2_small.npz", allow_pickle=False) as data:
    radio_X_test, radio_y_test = data["X_test"], data["y_test"]
# TODO 8
raise NotImplementedError("TODO 8")

assert radio_test_scores.shape == radio_test_flags.shape == (900,)
assert radio_test_flags.dtype == np.bool_
assert np.isfinite(radio_test_scores).all()
assert sum(radio_test_metrics[k] for k in ("tn", "fp", "fn", "tp")) == 900
print("Radio test results:", radio_test_metrics)
fig, ax = plt.subplots(figsize=(7, 3))
for label, name, colour in [(0, "Background", "#4978a5"), (1, "Pulsar", "#be593d")]:
    selected = np.flatnonzero(radio_y_test == label)
    ax.scatter(selected, radio_test_scores[selected], s=13, label=name, color=colour)
ax.axhline(tau, color="black", linestyle="--", label="Fixed threshold")
ax.set(xlabel="Test row", ylabel="Negative log density")
ax.legend()
fig.tight_layout()
plt.show()


**R2 (5 marks).**

(a) Report test FP, FN, background false-alarm rate, and pulsar recall.

(b) What do a false positive and a false negative mean for this screening task?

**Answer:**


## C. Fashion-MNIST: a small image classifier — 35 marks

Classify 28 × 28 grayscale images as 0 = T-shirt/top, 1 = Trouser,
or 2 = Bag. Training and validation samples come from the original training
set; the test sample comes from the original test set.

The training budget is fixed: 10 epochs, batch size 64, Adam with learning
rate 0.003. The supplied helper restores the epoch with the lowest validation CE.


In [ ]:
CLASS_NAMES = ["T-shirt/top", "Trouser", "Bag"]
with np.load(DATA_DIR / "fashion_small.npz", allow_pickle=False) as data:
    fashion_raw_train, fashion_raw_val = data["images_train"], data["images_val"]
    fashion_labels_train, fashion_labels_val = data["labels_train"], data["labels_val"]
assert fashion_raw_train.shape == (1200, 28, 28)
assert fashion_raw_val.shape == (300, 28, 28)
fig, axes = plt.subplots(1, 3, figsize=(6, 2.2))
for label, ax in enumerate(axes):
    i = np.flatnonzero(fashion_labels_train == label)[0]
    ax.imshow(fashion_raw_train[i], cmap="gray", vmin=0, vmax=255)
    ax.set_title(CLASS_NAMES[label])
    ax.axis("off")
fig.tight_layout()
plt.show()


**TODO 9 (5 marks):** convert the image arrays and class labels to tensors
for the CNN.
Given uint8 raw_images of shape (N, 28, 28)
and labels of shape (N,), **return (X, y)**.
X is a torch.float32 tensor of shape (N, 1, 28, 28), with pixel values
divided by 255. y is a torch.long tensor of shape (N,).


In [ ]:
def image_tensors(raw_images, labels):
    # TODO 9
    raise NotImplementedError("TODO 9")

check_X, check_y = image_tensors(
    np.full((2, 28, 28), 255, dtype=np.uint8), np.array([0, 2]))
assert check_X.shape == (2, 1, 28, 28) and torch.all(check_X == 1)
assert check_X.dtype == torch.float32 and check_y.dtype == torch.long
assert check_y.shape == (2,) and check_y.tolist() == [0, 2]

fashion_X_train, fashion_y_train = image_tensors(fashion_raw_train, fashion_labels_train)
fashion_X_val, fashion_y_val = image_tensors(fashion_raw_val, fashion_labels_val)
train_loader = DataLoader(
    TensorDataset(fashion_X_train, fashion_y_train), batch_size=64, shuffle=True,
    generator=torch.Generator().manual_seed(SEED))
print("Training inputs and targets:", fashion_X_train.shape, fashion_y_train.shape)


**TODO 10 (8 marks):** build and return a new, untrained nn.Sequential network with these layers:

| Layer | Settings |
|---|---|
| Conv2d | 1 input channel, 8 output channels, kernel 3, padding 1 |
| ReLU | |
| MaxPool2d | kernel 2 |
| Conv2d | 8 input channels, 16 output channels, kernel 3, padding 1 |
| ReLU | |
| MaxPool2d | kernel 2 |
| Flatten | |
| Linear | 16 × 7 × 7 inputs, 3 outputs |

Use default stride and bias settings. The returned model maps input tensors
of shape (N, 1, 28, 28) to raw class scores of shape (N, 3).


In [ ]:
def make_cnn():
    # TODO 10
    raise NotImplementedError("TODO 10")

torch.manual_seed(SEED)
shape_model = make_cnn()
assert isinstance(shape_model, nn.Sequential)
assert shape_model(torch.zeros(5, 1, 28, 28)).shape == (5, 3)
assert sum(p.numel() for p in shape_model.parameters()) == 3603
print("Batch shape after each layer:")
check_batch = torch.zeros(64, 1, 28, 28)
for layer in shape_model:
    check_batch = layer(check_batch)
    print(type(layer).__name__, tuple(check_batch.shape))


**S3 (5 marks).**

(a) For a batch of 64 images, state the shapes after the first convolution,
the first pooling layer, the second pooling layer, and the final linear layer.

(b) How does a convolutional filter share its weights across image positions?

**Answer:**


### Supplied training helper
The two losses below evaluate the full training and validation samples
using the weights at the end of each epoch.


In [ ]:
def cnn_loss(model, X, y):
    model.eval()
    with torch.no_grad():
        return float(nn.functional.cross_entropy(model(X), y))

def fit_cnn(model, loader, X_train, y_train, X_val, y_val):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
    history = {"train_ce": [], "val_ce": [], "best_epoch": None}
    best_loss, best_state = float("inf"), None
    for epoch in range(1, 11):
        model.train()
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            loss = nn.functional.cross_entropy(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        train_ce = cnn_loss(model, X_train, y_train)
        val_ce = cnn_loss(model, X_val, y_val)
        history["train_ce"].append(train_ce)
        history["val_ce"].append(val_ce)
        if val_ce < best_loss:
            best_loss = val_ce
            best_state = copy.deepcopy(model.state_dict())
            history["best_epoch"] = epoch
    model.load_state_dict(best_state)
    return history


**TODO 11 (4 marks):** create a fresh model with make_cnn, and call fit_cnn
with the supplied loader and training/validation tensors.

**Define:** fashion_model, the model restored by fit_cnn, and
fashion_history, its returned dictionary with train_ce, val_ce, and best_epoch.


In [ ]:
torch.manual_seed(SEED)
# TODO 11
raise NotImplementedError("TODO 11")

assert len(fashion_history["train_ce"]) == len(fashion_history["val_ce"]) == 10
assert fashion_history["best_epoch"] == int(np.argmin(fashion_history["val_ce"])) + 1
assert np.isclose(cnn_loss(fashion_model, fashion_X_val, fashion_y_val),
                  min(fashion_history["val_ce"]), atol=1e-7)
print("Selected epoch:", fashion_history["best_epoch"])
print("Best validation CE:", min(fashion_history["val_ce"]))
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(range(1, 11), fashion_history["train_ce"], "o-", label="Training")
ax.plot(range(1, 11), fashion_history["val_ce"], "o-", label="Validation")
ax.axvline(fashion_history["best_epoch"], color="gray", linestyle=":", label="Selected epoch")
ax.set(xlabel="Epoch", ylabel="Mean cross-entropy")
ax.set_xticks(range(1, 11))
ax.legend()
fig.tight_layout()
plt.show()


### Evaluate the selected checkpoint
**TODO 12 (6 marks):** evaluate the selected CNN on the test images.
Within the supplied evaluation block, compute raw class scores, test CE,
predictions using argmax over the class dimension, and accuracy.

Use the restored fashion_model for this final evaluation.

**Define these four variables:**

| Variable | Required result |
|---|---|
| fashion_test_logits | PyTorch tensor of shape (600, 3), raw class scores |
| fashion_test_ce | Scalar test cross-entropy |
| fashion_test_predictions | Integer PyTorch tensor of shape (600,), predicted classes |
| fashion_test_accuracy | Scalar proportion correct, between 0 and 1 |


In [ ]:
with np.load(DATA_DIR / "fashion_small.npz", allow_pickle=False) as data:
    fashion_raw_test, fashion_labels_test = data["images_test"], data["labels_test"]
fashion_X_test, fashion_y_test = image_tensors(fashion_raw_test, fashion_labels_test)
fashion_model.eval()
with torch.no_grad():
    # TODO 12
    raise NotImplementedError("TODO 12")

assert fashion_test_logits.shape == (600, 3)
assert fashion_test_predictions.shape == (600,)
assert torch.isfinite(fashion_test_logits).all()
assert 0 <= fashion_test_accuracy <= 1 and np.isfinite(fashion_test_ce)
print(f"Fashion test CE: {fashion_test_ce:.4f}")
print(f"Fashion test accuracy: {fashion_test_accuracy:.2%}")

fashion_confusion = np.zeros((3, 3), dtype=int)
np.add.at(fashion_confusion,
          (fashion_labels_test, fashion_test_predictions.cpu().numpy()), 1)
fig, ax = plt.subplots(figsize=(4.3, 3.4))
ax.imshow(fashion_confusion, cmap="Blues")
ax.set_xticks(range(3), CLASS_NAMES, rotation=15)
ax.set_yticks(range(3), CLASS_NAMES)
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(fashion_confusion[i, j]), ha="center", va="center",
                color="white" if fashion_confusion[i, j] > 100 else "black")
ax.set(xlabel="Predicted class", ylabel="Observed class", title="Test confusion matrix")
fig.tight_layout()
plt.show()

fashion_error_indices = np.flatnonzero(
    fashion_test_predictions.cpu().numpy() != fashion_labels_test)[:6]
if len(fashion_error_indices):
    fig, axes = plt.subplots(1, len(fashion_error_indices),
                             figsize=(2 * len(fashion_error_indices), 2.5), squeeze=False)
    for ax, i in zip(axes.flat, fashion_error_indices):
        pred = int(fashion_test_predictions[i])
        ax.imshow(fashion_raw_test[i], cmap="gray", vmin=0, vmax=255)
        ax.set_title(f"True: {CLASS_NAMES[int(fashion_labels_test[i])]}\nPred: {CLASS_NAMES[pred]}",
                     fontsize=9)
        ax.axis("off")
    fig.tight_layout()
    plt.show()
else:
    print("No test errors in this run.")


**R3 (7 marks).**

(a) Using fashion_history, report the epoch and CE at the minimum training CE
and at the minimum validation CE. Which checkpoint should be used for final
test predictions, and why?

(b) Treat Bag as the positive class and the other classes as negative.
Calculate precision and recall from the test confusion matrix.
Are false Bag predictions or missed Bags more frequent?
Support your answer with the counts.

**Answer:**


## Submission

Submit **one file: executed.pdf**, with every page in A4 format.
Include the complete notebook in order: code, outputs, figures, and S/R answers.

1. Complete the TODOs and answers. Restart the kernel, run all cells, and save.
2. Export the notebook to HTML with code and outputs visible, saving it as
   individual_project_3.html in the project folder, beside prepare_pdf.py.
3. Open a terminal in that folder and run:

       python3 prepare_pdf.py ./individual_project_3.html

4. Open the generated executed.html and print to PDF: A4, 100% scale,
   default margins, browser headers and footers off.
5. Inspect the saved PDF for complete, readable code, results, and figures.
   Submit executed.pdf and keep your notebook locally.

**Data attribution:** HTRU2: Lyon (2015), [UCI](https://doi.org/10.24432/C5DK6R),
CC BY 4.0. Fashion-MNIST: Xiao, Rasul, and Vollgraf (2017),
[Zalando Research](https://github.com/zalandoresearch/fashion-mnist), MIT.
See the data guide and attribution file for details.
